# Graph End-to-End Validation Notebook
Este notebook executa os testes de uso das classes novas em sequência:
1. Carrega `.env` e autentica no Graph
2. Obtém e apresenta o conteúdo do site
3. Lista arquivos do drive e salva em DataFrame (`df_drive_items`)
4. Executa fluxo de arquivo: write, update e load
5. Executa testes de listas (views, colunas, itens, create e update)

In [7]:
from __future__ import annotations
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv

# Garante carregamento do .env a partir da raiz do repositório
repo_root = Path.cwd()
if not (repo_root / '.env').exists() and (repo_root.parent / '.env').exists():
    repo_root = repo_root.parent

env_path = repo_root / '.env'
load_dotenv(env_path)

print(f'.env carregado de: {env_path}')
print('Pronto para iniciar os testes.')

.env carregado de: c:\GitHub\MSGraphClient\.env
Pronto para iniciar os testes.


In [8]:
from python.auth import GraphClient
from python.drive import GraphDrive
from python.lists import GraphList

client = GraphClient()
auth = client.authenticator
print('Cliente Graph inicializado com sucesso.')

site_attributes = {
    'sharepoint_site_id': auth.sharepoint_site_id,
    'site_graph_id': auth.site_graph_id,
    'site_name': auth.site_name,
    'site_display_name': auth.site_display_name,
    'site_web_url': auth.site_web_url,
}

print('\nAtributos do site conectado:')
for key, value in site_attributes.items():
    print(f'- {key}: {value}')

Cliente Graph inicializado com sucesso.

Atributos do site conectado:
- sharepoint_site_id: anatel365.sharepoint.com,d8124f53-4224-4434-b3ec-97f23a341b20,7fad1d86-65ba-4c88-94b9-2001d3f91ff2
- site_graph_id: anatel365.sharepoint.com,d8124f53-4224-4434-b3ec-97f23a341b20,7fad1d86-65ba-4c88-94b9-2001d3f91ff2
- site_name: sfi_app_private
- site_display_name: SFI App Dados Restritos
- site_web_url: https://anatel365.sharepoint.com/sites/sfi_app_private


In [9]:
site_contents = auth.get_site_contents()
site_data = site_contents.get('site', {})
drives_data = site_contents.get('drives', [])
lists_data = site_contents.get('lists', [])

print('Resumo do conteúdo do site:')
print(f"- Site: {site_data.get('displayName', site_data.get('name', '-'))}")
print(f"- Drives encontrados: {len(drives_data)}")
print(f"- Lists encontradas: {len(lists_data)}")

df_site_drives = pd.json_normalize(drives_data) if drives_data else pd.DataFrame()
df_site_lists = pd.json_normalize(lists_data) if lists_data else pd.DataFrame()

print('\nDrives Disponíveis:')
display(df_site_drives.head())

print('Lists Disponíveis:')
display(df_site_lists.head())

Resumo do conteúdo do site:
- Site: SFI App Dados Restritos
- Drives encontrados: 2
- Lists encontradas: 4

Drives Disponíveis:


,id,name,webUrl,driveType
0,b!U08S2CRCNESz7JfyOjQbIIYdrX-6ZYhMlLkgAdP5H_Lp...,Documentos,https://anatel365.sharepoint.com/sites/sfi_app...,documentLibrary
1,b!U08S2CRCNESz7JfyOjQbIIYdrX-6ZYhMlLkgAdP5H_I0...,GetMonitorRNI,https://anatel365.sharepoint.com/sites/sfi_app...,documentLibrary


Lists Disponíveis:


,@odata.etag,id,name,webUrl,displayName
0,"""a0db896a-d211-455c-a833-045ce1c7c40f,0""",a0db896a-d211-455c-a833-045ce1c7c40f,wte,https://anatel365.sharepoint.com/sites/sfi_app...,Extensões de Modelo da Web
1,"""4e8c9d10-db78-42b8-bdbc-3f44f1a6ce98,4""",4e8c9d10-db78-42b8-bdbc-3f44f1a6ce98,monitorRNI,https://anatel365.sharepoint.com/sites/sfi_app...,monitorRNI
2,"""7384fae9-17d5-40af-ba50-9adc5816a09b,6""",7384fae9-17d5-40af-ba50-9adc5816a09b,Documentos Compartilhados,https://anatel365.sharepoint.com/sites/sfi_app...,Documentos
3,"""a9d8be34-b989-483a-a308-e7529a99c623,10""",a9d8be34-b989-483a-a308-e7529a99c623,GetMonitorRNI,https://anatel365.sharepoint.com/sites/sfi_app...,GetMonitorRNI


In [10]:
drive = GraphDrive(client=client)
drive_items = drive.list_drive_items()

df_drive_items = pd.json_normalize(drive_items) if drive_items else pd.DataFrame()
num_files = len([item for item in drive_items if 'folder' not in item])

print(f'Total de itens no drive root: {len(drive_items)}')
print(f'Total de arquivos (sem pastas): {num_files}')
print('\nDados disponíveis:')

display(df_drive_items.head())

Total de itens no drive root: 10
Total de arquivos (sem pastas): 10

Dados disponíveis:


,@microsoft.graph.downloadUrl,createdDateTime,eTag,id,lastModifiedDateTime,name,webUrl,cTag,isAuthoritative,size,...,file.fileExtension,file.hashes.quickXorHash,file.mimeType,fileSystemInfo.createdDateTime,fileSystemInfo.lastModifiedDateTime,shared.scope,createdBy.user.email,createdBy.user.id,lastModifiedBy.user.email,lastModifiedBy.user.id
0,https://anatel365.sharepoint.com/sites/sfi_app...,2026-05-26T15:16:02Z,"""{3E8865CC-1A7D-4A50-BFAC-4253FBE37488},2""",01ZUTETVOMMWED47I2KBFL7LCCKP56G5EI,2026-05-26T15:16:05Z,notebook_test_upload_20260526_151559.txt,https://anatel365.sharepoint.com/sites/sfi_app...,"""c:{3E8865CC-1A7D-4A50-BFAC-4253FBE37488},2""",False,106,...,.txt,GvFPOUfa1FqFvTedLbv/4MGiqrg=,text/plain,2026-05-26T15:16:02Z,2026-05-26T15:16:05Z,users,NaN,NaN,NaN,NaN
1,https://anatel365.sharepoint.com/sites/sfi_app...,2026-05-26T15:41:17Z,"""{47F2A2E1-0536-4675-AD3C-DEE010F1C3ED},2""",01ZUTETVPBULZEONQFOVDK2PG64AIPDQ7N,2026-05-26T15:41:19Z,notebook_test_upload_20260526_154114.txt,https://anatel365.sharepoint.com/sites/sfi_app...,"""c:{47F2A2E1-0536-4675-AD3C-DEE010F1C3ED},2""",False,106,...,.txt,CvFNOYfa0WoFvzsNLbv/4Oliq7g=,text/plain,2026-05-26T15:41:17Z,2026-05-26T15:41:19Z,users,NaN,NaN,NaN,NaN
2,https://anatel365.sharepoint.com/sites/sfi_app...,2026-05-26T16:40:44Z,"""{038461E9-EEA3-4FFE-B0EB-BC944DC3F9D5},2""",01ZUTETVPJMGCAHI7O7ZH3B254SRG4H6OV,2026-05-26T16:40:46Z,notebook_test_upload_20260526_164041.txt,https://anatel365.sharepoint.com/sites/sfi_app...,"""c:{038461E9-EEA3-4FFE-B0EB-BC944DC3F9D5},2""",False,106,...,.txt,WnFPOWfa0lKFvDHdLdv/4Okiq7g=,text/plain,2026-05-26T16:40:44Z,2026-05-26T16:40:46Z,users,NaN,NaN,NaN,NaN
3,https://anatel365.sharepoint.com/sites/sfi_app...,2026-05-26T17:01:13Z,"""{FA529733-0F79-4A39-9996-87746221E24E},2""",01ZUTETVJTS5JPU6IPHFFJTFUHORRCDYSO,2026-05-26T17:01:15Z,notebook_test_upload_20260526_170110.txt,https://anatel365.sharepoint.com/sites/sfi_app...,"""c:{FA529733-0F79-4A39-9996-87746221E24E},2""",False,106,...,.txt,CvFPOYfa1FJFvTkdLfv/4Mliq7g=,text/plain,2026-05-26T17:01:13Z,2026-05-26T17:01:15Z,users,NaN,NaN,NaN,NaN
4,https://anatel365.sharepoint.com/sites/sfi_app...,2026-05-27T15:03:41Z,"""{DF1DC1A3-8E03-4DD7-869D-AAE4A00E2C43},2""",01ZUTETVNDYEO56A4O25GYNHNK4SQA4LCD,2026-05-27T15:03:43Z,notebook_test_upload_20260527_150339.txt,https://anatel365.sharepoint.com/sites/sfi_app...,"""c:{DF1DC1A3-8E03-4DD7-869D-AAE4A00E2C43},2""",False,106,...,.txt,WnFOOSfa0WpFrDGNLbv/4Mniq7g=,text/plain,2026-05-27T15:03:41Z,2026-05-27T15:03:43Z,users,lobao@anatel.gov.br,26fdc052-67a9-4c27-86ce-c541f3424d6f,lobao@anatel.gov.br,26fdc052-67a9-4c27-86ce-c541f3424d6f


## File Write, Update and Load
Este bloco cria (ou reutiliza) um arquivo de teste, atualiza o conteúdo e lê novamente para validação.

In [11]:
test_local_file = repo_root / 'notebooks' / 'downloads' / 'notebook_test_upload.txt'
test_local_file.parent.mkdir(parents=True, exist_ok=True)
test_local_file.write_text(
    f'Notebook initial content - {datetime.now(timezone.utc).isoformat()}\\n',
    encoding='utf-8',
)

upload_result = drive.upload_file(
    local_path=test_local_file,
    remote_folder='root',
    remote_name=f'notebook_test_upload_{datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")}.txt',
)
target_item_id = str(upload_result.get('id', ''))

if not target_item_id:
    raise RuntimeError('Falha ao obter item id após upload do arquivo de teste.')

print('Write concluído (upload):')
print(f"- Item ID: {target_item_id}")
print(f"- Nome: {upload_result.get('name')}")

updated_content = (
    f'Notebook updated content - {datetime.now(timezone.utc).isoformat()}\\n'
    'Linha adicional para teste de update.\\n'
    'Fim.\\n'
 )
update_result = drive.write_file_content(target_item_id, updated_content)

print('Update concluído:')
print(f"- Item ID retornado: {update_result.get('id')}")

loaded_content = drive.read_file_content(target_item_id)
print('Load concluído (conteúdo atual lido):')
print(loaded_content)

download_path = repo_root / 'notebooks' / 'downloads' / f'downloaded_{target_item_id}.txt'
saved_path = drive.download_file(target_item_id, download_path)
print(f'Arquivo carregado para disco local em: {saved_path}')

Write concluído (upload):
- Item ID: 01ZUTETVNJKDHT2AON75AKSVT3ESEH7ONM
- Nome: notebook_test_upload_20260527_183047.txt
Update concluído:
- Item ID retornado: 01ZUTETVNJKDHT2AON75AKSVT3ESEH7ONM
Load concluído (conteúdo atual lido):
Notebook updated content - 2026-05-27T18:30:48.785928+00:00\nLinha adicional para teste de update.\nFim.\n
Arquivo carregado para disco local em: C:\GitHub\MSGraphClient\notebooks\downloads\downloaded_01ZUTETVNJKDHT2AON75AKSVT3ESEH7ONM.txt


## List Tests (Guided)
Este bloco agora segue um fluxo guiado:
1. Inicializa `GraphList` com compatibilidade de versão da API.
2. Exibe schema, tipos e templates de payload (fáceis de copiar/colar e editar).
3. Permite salvar um item como `create` ou `update` em uma célula separada.

In [ ]:
import importlib
import json
import python.lists as lists_mod

# Recarrega módulo para evitar cache de versões anteriores no kernel.
importlib.reload(lists_mod)
GraphList = lists_mod.GraphList

list_client = GraphList(client=client)

# Compatibilidade entre nomes novos e antigos (caso o ambiente esteja desatualizado).
get_views_fn = getattr(list_client, 'get_views', None) or getattr(list_client, 'get_list_views')
get_columns_fn = getattr(list_client, 'get_columns', None) or getattr(list_client, 'get_list_columns')
get_view_columns_fn = getattr(list_client, 'get_view_columns', None) or getattr(list_client, 'get_list_view_columns', None)

views = get_views_fn()
columns = get_columns_fn()
schema = list_client.get_schema()
field_types = list_client.get_field_types()

print('GraphList inicializado com sucesso.')
print(f'- Método views ativo: {get_views_fn.__name__}')
print(f'- Método columns ativo: {get_columns_fn.__name__}')
if get_view_columns_fn:
    print(f'- Método view_columns ativo: {get_view_columns_fn.__name__}')

print(f'Views encontradas: {len(views)}')
print(f'Colunas consultadas: {len(columns)}')
print(f'Colunas no schema validado: {len(schema)}')

df_schema = pd.DataFrame(schema) if schema else pd.DataFrame()
print('\nSchema validado da lista (display_name, type, required, read_only):')
display(df_schema)

df_list_items = list_client.get_items_dataframe(include_id=True)
print(f'Itens atuais encontrados: {len(df_list_items)}')
display(df_list_items.head())

schema_by_name = {entry['display_name']: entry for entry in schema}
now = datetime.now(timezone.utc)

def first_writable_of_type(field_type: str) -> str | None:
    for entry in schema:
        if entry.get('read_only'):
            continue
        if entry.get('type') == field_type:
            return str(entry['display_name'])
    return None

# Template mínimo para create (campos obrigatórios).
suggested_payload_create = list_client.get_item_template(include_optional=False)
for display_name in list(suggested_payload_create.keys()):
    col_type = field_types.get(display_name)
    if col_type in ('text', 'note', 'personOrGroup'):
        suggested_payload_create[display_name] = f'Novo item - {now.strftime("%Y-%m-%d %H:%M:%S")}'
    elif col_type == 'number':
        suggested_payload_create[display_name] = 0
    elif col_type == 'boolean':
        suggested_payload_create[display_name] = False
    elif col_type == 'dateTime':
        suggested_payload_create[display_name] = now.isoformat()
    elif col_type == 'choice':
        choices = schema_by_name.get(display_name, {}).get('choices', [])
        suggested_payload_create[display_name] = choices[0] if choices else None

# Template exemplo para update (usa primeiro _id encontrado, se existir).
suggested_payload_update = {'_id': str(df_list_items.iloc[0]['_id'])} if not df_list_items.empty else {'_id': '<INSIRA_O_ID_AQUI>'}
text_col = first_writable_of_type('text')
if text_col:
    suggested_payload_update[text_col] = f'Atualizado via notebook em {now.isoformat()}'
number_col = first_writable_of_type('number')
if number_col:
    suggested_payload_update[number_col] = 123.45
boolean_col = first_writable_of_type('boolean')
if boolean_col:
    suggested_payload_update[boolean_col] = True
date_col = first_writable_of_type('dateTime')
if date_col:
    suggested_payload_update[date_col] = now.isoformat()
choice_col = first_writable_of_type('choice')
if choice_col:
    choices = schema_by_name.get(choice_col, {}).get('choices', [])
    if choices:
        suggested_payload_update[choice_col] = choices[0]

print('\nTemplate sugerido para CREATE (copie e edite na próxima célula):')
print(json.dumps(suggested_payload_create, ensure_ascii=False, indent=2, default=str))

print('\nTemplate sugerido para UPDATE (copie e edite na próxima célula):')
print(json.dumps(suggested_payload_update, ensure_ascii=False, indent=2, default=str))

AttributeError: 'GraphList' object has no attribute 'get_views'

### Save Operation (Editável)
Edite os valores da próxima célula e execute:
- `operation = 'create'` para inserir novo item.
- `operation = 'update'` para atualizar item existente (exige `_id`).

In [ ]:
# Escolha: 'create' ou 'update'
operation = 'update'

# Copie um dos templates exibidos na célula anterior e edite aqui.
payload = dict(suggested_payload_update)

# Exemplo opcional para create:
# operation = 'create'
# payload = dict(suggested_payload_create)

if operation not in ('create', 'update'):
    raise ValueError("operation deve ser 'create' ou 'update'")

if operation == 'create':
    payload.pop('_id', None)
else:
    if '_id' not in payload or not str(payload['_id']).strip():
        raise ValueError("Para update, informe payload['_id'] com o ID do item")

print('Validando payload...')
list_client.validate_item(payload)

print(f'Salvando item ({operation})...')
saved_item = list_client.save_item(payload)
print('Operação concluída com sucesso:')
print(saved_item)

print('\nAmostra atualizada de itens:')
df_list_items_after_save = list_client.get_items_dataframe(include_id=True)
display(df_list_items_after_save.head())